# Vietnamese Text Classification with Mixture of Experts experiment

This notebook prepares a **Kaggle Notebook runtime** to train and evaluate the ViHSD Mixture of Experts (MoE) architectures.

Training and evaluation are executed explicitly from shell commands. This keeps experiment execution reproducible, parameter overrides transparent, and run outputs systematically tracked by unique `run_id`s.

### Essential Kaggle Settings Checklist
Before running any cells, make sure these settings are enabled in the Kaggle right-hand **Notebook settings** sidebar:
1. **Accelerator**: Select **GPU P100** (single 16GB GPU) or **GPU T4 x2** (15GB VRAM per card).
2. **Internet**: Toggle **Internet ON** (requires SMS verification on Kaggle). Internet is required to clone this repository, download the ViHSD dataset from Hugging Face, download PhoBERT weights, and log to Weights & Biases.
3. **Secrets**: Under the notebook's top menu, open **Add-ons → Secrets** and create:
   - `HF_TOKEN`: Hugging Face access token.
   - `WANDB_API_KEY`: Weights & Biases API key.
4. **Output Persistence**: Everything written to `/kaggle/working` is saved permanently when you click **"Save Version" → "Save & Run All (Commit)"**. Completed runs will appear in the notebook's **Output** tab.

---

## 1. Configure Kaggle Output Directories

On Kaggle, `/kaggle/working` is the only writable directory during execution. We configure `CHECKPOINT_DIR` and `RESULTS_DIR` environment variables so all model checkpoints (`.safetensors`) and experiment logs (`run_metrics.json`, `vihsd_predictions.json`) are saved directly to `/kaggle/working/checkpoints` and `/kaggle/working/results`.

In [ ]:
import os
from pathlib import Path

checkpoint_dir = Path('/kaggle/working/checkpoints')
results_dir = Path('/kaggle/working/results')

checkpoint_dir.mkdir(parents=True, exist_ok=True)
results_dir.mkdir(parents=True, exist_ok=True)

os.environ['CHECKPOINT_DIR'] = str(checkpoint_dir)
os.environ['RESULTS_DIR'] = str(results_dir)

print(f"CHECKPOINT_DIR: {os.environ['CHECKPOINT_DIR']}")
print(f"RESULTS_DIR:    {os.environ['RESULTS_DIR']}")

## 2. Clone the Repository & Verify Dependencies

This step ensures the repository code is available in the workspace and targets the `main` branch.

If you are running this notebook inside an already cloned or uploaded workspace where `train.py` is present, it will automatically use the current directory without re-cloning.

In [ ]:
import os
from pathlib import Path

PROJECT_DIR = '/kaggle/working/moe-vihsd'
REPOSITORY_URL = 'https://github.com/lngphgthao/moe-vihsd.git'
BRANCH = 'main'

if not Path('train.py').exists():
    if not Path(PROJECT_DIR).exists():
        print(f"Cloning {REPOSITORY_URL} (branch: {BRANCH})...")
        !git clone --depth 1 --branch $BRANCH $REPOSITORY_URL $PROJECT_DIR
    %cd $PROJECT_DIR
else:
    print(f"Already in project repository root: {Path.cwd()}")

%pip install -q -r requirements.txt

## 3. Authenticate via Kaggle Secrets

Authentication credentials for Hugging Face and Weights & Biases are retrieved securely using Kaggle's `UserSecretsClient`.

Ensure you have added `HF_TOKEN` and `WANDB_API_KEY` under **Add-ons → Secrets** in the notebook toolbar.

In [ ]:
import os
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret('HF_TOKEN')
    wandb_api_key = user_secrets.get_secret('WANDB_API_KEY')
except Exception as error:
    print(f"Kaggle secrets client notice: {error}")
    hf_token = os.getenv('HF_TOKEN')
    wandb_api_key = os.getenv('WANDB_API_KEY')

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    print("✓ Hugging Face authentication configured.")
else:
    print("! Notice: HF_TOKEN secret not found; public Hugging Face assets will still load.")

if wandb_api_key:
    os.environ['WANDB_API_KEY'] = wandb_api_key
    wandb.login(key=wandb_api_key)
    print("✓ Weights & Biases authentication successful.")
else:
    print("! Notice: WANDB_API_KEY secret not found; set logging.use_wandb=false if W&B is not needed.")

## 4. Run Training and Evaluation Manually

The repository uses one shared training pipeline with four architecture families:

- `dense_transformer`: dense Hugging Face encoder and classifier.
- `transformer_moe`: selected internal Transformer FFNs replaced by token-level MoE.
- `dynamic_moe`: dense encoder with sequence-level dynamic-threshold routing.
- `classifier_moe`: dense encoder with a pooled-representation MoE classification head.

Use canonical architecture names for new runs. The older `dense_phobert` and `phobert_moe` names remain compatibility aliases. Each run saves its resolved configuration, checkpoint, metrics, and predictions. Use a unique `--run-id` for every experiment.

### Smoke test

```bash
python train.py --config configs/vihsd.yaml --smoke-test --run-id smoke-check
python evaluate.py --config configs/vihsd.yaml --run-id smoke-check
```

### Dense Transformer baseline

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id dense-transformer-integrated \
  --set model.architecture=dense_transformer \
  --set model.pooling=mean \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id dense-transformer-integrated
```

### Transformer-level MoE

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id transformer-moe-integrated \
  --set model.architecture=transformer_moe \
  --set training.loss_type=cross_entropy
python evaluate.py --config configs/vihsd.yaml --run-id transformer-moe-integrated
```

When debugging, inspect `checkpoints/<run-id>/resolved_config.yaml` before rerunning.

## 5. Training Command

Edit the command below and execute the cell manually. For a persistent run, save a Kaggle version via **Save Version → Save & Run All (Commit)**.

In [ ]:
# Edit the run_id before executing this cell.
# Dense Transformer baseline:
!python train.py --config configs/vihsd.yaml --no-smoke-test --run-id dense-transformer-integrated --set model.architecture=dense_transformer --set model.pooling=mean --set training.loss_type=cross_entropy

# Transformer-level MoE:
# !python train.py --config configs/vihsd.yaml --no-smoke-test --run-id transformer-moe-integrated --set model.architecture=transformer_moe --set training.loss_type=cross_entropy

## 6. Evaluation Command

Run evaluation after training finishes. Pass the matching `--run-id` to load that run's resolved configuration and best checkpoint. Evaluation writes classification metrics, predictions, confusion-matrix counts, and class-distribution diagnostics under `results/<run-id>/`.

In [ ]:
# Use the run_id from the training command above:
!python evaluate.py --config configs/vihsd.yaml --run-id transformer-moe-integrated

# Or evaluate a specific checkpoint directly:
# !python evaluate.py --config configs/vihsd.yaml --checkpoint /kaggle/working/checkpoints/transformer-moe-integrated/transformer_moe_best.safetensors

## 7. Command-line and configuration reference

Configuration roles and inheritance rules are documented in `configs/README.md`.

### Command-line arguments

| Command | Argument | Default | What it does |
| --- | --- | --- | --- |
| `train.py` | `--config PATH` | `configs/vihsd.yaml` | Loads the YAML configuration and optional `base_config`. |
| `train.py` | `--set SECTION.KEY=VALUE` | none | Overrides an existing YAML value; repeat it for multiple overrides. |
| `train.py` | `--smoke-test` / `--no-smoke-test` | config value (`false`) | Selects the short validation profile or full training. |
| `train.py` | `--run-id ID` | timestamped ID | Names the checkpoint and results directory. |
| `evaluate.py` | `--config PATH` | `configs/vihsd.yaml` | Supplies the configuration used to rebuild the model and data pipeline. |
| `evaluate.py` | `--run-id ID` | latest run | Evaluates a saved run using its resolved configuration. |
| `evaluate.py` | `--checkpoint PATH` | none | Evaluates one explicit `.safetensors` checkpoint. |

### Configuration arguments

These defaults come from `configs/vihsd.yaml`. Use `--set section.key=value` to override a value for one run.

| Key | Default | What it does |
| --- | --- | --- |
| `seed` | `42` | Seeds Python, NumPy, and PyTorch. |
| `dataset.name` | `data/vihsd_segmented` | Dataset path or Hugging Face dataset ID. |
| `dataset.max_length` | `128` | Maximum token sequence length. |
| `model.architecture` | `transformer_moe` | Selects `dense_transformer`, `transformer_moe`, `dynamic_moe`, or `classifier_moe`. |
| `model.pretrained_model_name` | `vinai/phobert-base` | Encoder and tokenizer model ID or local path. Change this for XLM-R or mBERT. |
| `model.moe_layers` | `[8, 9, 10, 11]` | Zero-based layers whose FFNs are replaced by Transformer MoE. |
| `model.num_experts` | `4` | Number of routed experts. |
| `model.top_k` | `1` | Experts selected for each token or example. |
| `model.routing_method` | `top_k` | Transformer-MoE routing method: `top_k` or `dynamic_threshold`. |
| `model.upcycle` | `true` | Initializes GELU experts from pretrained FFN weights. |
| `model.shared_expert` | `false` | Adds an always-active expert. |
| `model.expert_type` | `gelu` | Transformer expert type: `gelu`, `geglu`, or `swiglu`. |
| `model.freeze_backbone` | `false` | Freezes the complete encoder for dense or classifier-level MoE models. |
| `model.pooling` | `cls` | Uses the first token or mean of non-padding tokens. |
| `training.epochs` | `5` | Number of full epochs. |
| `training.batch_size` | `16` | Examples per optimization step. |
| `training.learning_rate` | `0.00002` | AdamW learning rate. |
| `training.weight_decay` | `0.01` | AdamW weight decay. |
| `training.loss_type` | `cross_entropy` | Task loss type. |
| `routing.load_balance_loss_factor` | `0.01` | Weight of the auxiliary MoE balancing loss. |
| `logging.use_wandb` | `true` | Enables W&B logging. |

The model name controls both encoder and tokenizer, so XLM-R requires only:

```bash
python train.py --config configs/vihsd.yaml --no-smoke-test \
  --run-id xlmr-transformer-moe-s42 \
  --set model.architecture=transformer_moe \
  --set model.pretrained_model_name=xlm-roberta-base
```

For dense mBERT, use `model.architecture=dense_transformer`. Use the dedicated overlays under `configs/architectures/`, `configs/stages/`, and `configs/diagnostics/` when they match the experiment purpose.
